# P2 — Dataset Exploration

**Amaç:** `data_prep.py` tarafından üretilen Türkçe lise Q&A dataset'ini keşfedip
kalitesini görsel olarak değerlendirmek. Fine-tuning öncesi son kontrol noktası.

**Kontrol edilenler:**
- Toplam örnek sayısı + subject × grade dağılımı
- Output uzunluğu istatistikleri (karakter + token)
- Dataset dengesi (outlier, eksik subject/grade kombinasyonları)
- Tokenizer bazlı `max_seq_length=512` yeterli mi? (Qwen3-4B tokenizer)

**Çalıştırma:** Notebook `ml/notebooks/` dizininden çalıştırılır; path'ler `..`
üzerinden relative. Lokal'de `pip install pandas matplotlib pyyaml transformers`
veya Colab'da `pip install -r requirements.txt` yeterli.

> `nbstripout` aktif: output'lar commit'te otomatik temizlenir.
> Lokalde "Run All" yapıp gözlem için saklayabilirsin.

In [ ]:
# Veri yükleme — train.jsonl satır satır JSON parse, pandas DataFrame'e çevir
import json
from pathlib import Path

import pandas as pd

# Notebook ml/notebooks/ içinden çalışıyor — proje kökü iki seviye yukarıda
ML_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("ml")
TRAIN_PATH = ML_ROOT / "data" / "processed" / "train.jsonl"

records = [
    json.loads(line)
    for line in TRAIN_PATH.read_text(encoding="utf-8").splitlines()
]
df = pd.DataFrame(records)
print(f"Yüklenen kayıt: {len(df)} | Sütunlar: {list(df.columns)}\n")
df.head(5)

In [ ]:
# Temel istatistikler — sayım, dağılım, uzunluk profili
print(f"Toplam örnek:     {len(df)}")
print(f"Eşsiz subject:    {df['subject'].nunique()}")
print(f"Eşsiz grade:      {df['grade'].nunique()}\n")

print("=== Subject Dağılımı ===")
print(df["subject"].value_counts().to_string())

print("\n=== Grade Dağılımı ===")
print(df["grade"].value_counts().sort_index().to_string())

# Karakter bazlı uzunluk; token analizi Hücre 6'da
char_lens = df["output"].str.len()
print(f"\n=== Output Uzunluğu (karakter) ===")
print(f"Ortalama: {char_lens.mean():.0f}")
print(f"Medyan:   {char_lens.median():.0f}")
print(f"Min/Max:  {char_lens.min()} / {char_lens.max()}")
print(f"Std:      {char_lens.std():.0f}")

In [ ]:
# Subject × Grade dağılımı — heatmap + stacked bar (iki açıdan dengelilik kontrolü)
import matplotlib.pyplot as plt

# Pivot table: satır=subject, sütun=grade, değer=örnek sayısı
pivot = df.pivot_table(
    index="subject", columns="grade", aggfunc="size", fill_value=0
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Sol: Heatmap — yoğunluk haritası
im = axes[0].imshow(pivot.values, cmap="YlOrRd", aspect="auto")
axes[0].set_xticks(range(len(pivot.columns)))
axes[0].set_xticklabels(pivot.columns)
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels(pivot.index)
axes[0].set_xlabel("Grade")
axes[0].set_ylabel("Subject")
axes[0].set_title("Subject × Grade Heatmap (count)")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        axes[0].text(
            j, i, pivot.values[i, j],
            ha="center", va="center", color="black", fontsize=10,
        )
fig.colorbar(im, ax=axes[0], label="Örnek sayısı")

# Sağ: Stacked bar — subject toplamı, grade rengi
pivot.plot(kind="bar", stacked=True, ax=axes[1], colormap="tab10", edgecolor="white")
axes[1].set_xlabel("Subject")
axes[1].set_ylabel("Örnek sayısı")
axes[1].set_title("Subject Toplamları (grade kırılımı renkli)")
axes[1].legend(title="Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Sayısal özet
print(f"\nSubject sayım aralığı: {pivot.sum(axis=1).min()} - {pivot.sum(axis=1).max()}")
print(f"Grade sayım aralığı:   {pivot.sum(axis=0).min()} - {pivot.sum(axis=0).max()}")
print(f"Boş hücre (subject × grade kombinasyonu yok): {(pivot == 0).sum().sum()}")

In [ ]:
# Output uzunluğu histogramı — outlier kontrolü (çok kısa veya çok uzun cevaplar)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sol: Histogram — dağılım şekli
axes[0].hist(char_lens, bins=30, color="steelblue", edgecolor="black", alpha=0.75)
axes[0].axvline(char_lens.mean(), color="red", linestyle="--", label=f"Ort: {char_lens.mean():.0f}")
axes[0].axvline(char_lens.median(), color="orange", linestyle="--", label=f"Medyan: {char_lens.median():.0f}")
axes[0].set_xlabel("Output uzunluğu (karakter)")
axes[0].set_ylabel("Örnek sayısı")
axes[0].set_title(f"Output Uzunluğu Dağılımı (n={len(df)})")
axes[0].legend()

# Sağ: Scatter — outlier index'leri görünür
SHORT_THRESHOLD = 100
LONG_THRESHOLD = 900
colors = [
    "red" if (l < SHORT_THRESHOLD or l > LONG_THRESHOLD) else "steelblue"
    for l in char_lens
]
axes[1].scatter(range(len(char_lens)), char_lens, c=colors, s=12, alpha=0.7)
axes[1].axhline(SHORT_THRESHOLD, color="orange", linestyle="--", alpha=0.5,
                label=f"<{SHORT_THRESHOLD} (kısa)")
axes[1].axhline(LONG_THRESHOLD, color="orange", linestyle="--", alpha=0.5,
                label=f">{LONG_THRESHOLD} (uzun)")
axes[1].set_xlabel("Örnek index")
axes[1].set_ylabel("Karakter")
axes[1].set_title("Outlier Konumu (kırmızı = sınır dışı)")
axes[1].legend()

plt.tight_layout()
plt.show()

# Outlier sayım — data_prep.py filter (20-1000 char) ile tutarlılık kontrolü
n_short = (char_lens < SHORT_THRESHOLD).sum()
n_long = (char_lens > LONG_THRESHOLD).sum()
n_filter_violation = ((char_lens < 20) | (char_lens > 1000)).sum()
print(f"\nKısa (<{SHORT_THRESHOLD} chr):  {n_short:>3} örnek")
print(f"Uzun (>{LONG_THRESHOLD} chr):  {n_long:>3} örnek")
print(f"data_prep filter ihlali (20-1000 dışı): {n_filter_violation} "
      f"(beklenen: 0 — filter doğru çalıştıysa)")

In [ ]:
# Tokenizer analizi — Qwen3-4B-Instruct-2507 tokenizer ile token sayımı
# max_seq_length=512 yeterli mi? Truncation gerekir mi?
import yaml
from transformers import AutoTokenizer

# Config'ten model adını oku — base model değişirse otomatik adapte olur
config = yaml.safe_load((ML_ROOT / "training" / "config.yaml").read_text())
MODEL_NAME = config["model"]["name"]
MAX_SEQ = config["model"]["max_length"]

print(f"Model:           {MODEL_NAME}")
print(f"max_seq_length:  {MAX_SEQ}")
print("\nTokenizer indiriliyor (~7 MB, ilk çalıştırmada cache'e yazılır)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text: str) -> int:
    """Special token'lar olmadan saf text token sayısı."""
    return len(tokenizer.encode(text, add_special_tokens=False))

# Output + instruction token sayımları
df["output_tokens"] = df["output"].apply(count_tokens)
df["instruction_tokens"] = df["instruction"].apply(count_tokens)
# +20 chat template overhead (Qwen3 ChatML: <|im_start|>user\n...<|im_end|>...)
df["total_tokens"] = df["output_tokens"] + df["instruction_tokens"] + 20

print(f"\n=== Output Token Dağılımı ===")
print(df["output_tokens"].describe().round(0).astype(int).to_string())

# max_seq_length kontrolü — kaç örnek truncation gerektirir?
over_limit = (df["total_tokens"] > MAX_SEQ).sum()
near_limit = ((df["total_tokens"] > MAX_SEQ * 0.9) & (df["total_tokens"] <= MAX_SEQ)).sum()
print(f"\n=== Sequence Length Analizi ===")
print(f"max_seq_length ({MAX_SEQ}) aşan örnek:  {over_limit:>3} / {len(df)}")
print(f"Sınıra yakın (>%90):             {near_limit:>3} / {len(df)}")
if over_limit > 0:
    print(f"\n⚠ Truncation tetiklenecek:")
    print(df[df["total_tokens"] > MAX_SEQ][
        ["subject", "grade", "total_tokens"]
    ].head().to_string())
else:
    print("✓ Tüm örnekler max_seq_length altında — truncation yok")

# En uzun 3 örnek
print("\n=== En Uzun 3 Örnek ===")
for _, row in df.nlargest(3, "output_tokens").iterrows():
    print(f"\n[{row['subject']} grade {row['grade']}] output={row['output_tokens']} tok, total={row['total_tokens']} tok")
    print(f"  Q: {row['instruction'][:90]}...")
    print(f"  A: {row['output'][:130]}...")

# En kısa 3 örnek — minimum kalite kontrolü
print("\n=== En Kısa 3 Örnek ===")
for _, row in df.nsmallest(3, "output_tokens").iterrows():
    print(f"\n[{row['subject']} grade {row['grade']}] output={row['output_tokens']} tok")
    print(f"  Q: {row['instruction']}")
    print(f"  A: {row['output']}")

# Token bazlı histogram — character bazlı'dan farklı görünebilir
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["output_tokens"], bins=30, color="forestgreen", edgecolor="black", alpha=0.75)
ax.axvline(df["output_tokens"].mean(), color="red", linestyle="--",
           label=f"Ort: {df['output_tokens'].mean():.0f} token")
ax.axvline(MAX_SEQ * 0.9, color="orange", linestyle=":", alpha=0.5,
           label=f"max_seq × 0.9 ({int(MAX_SEQ * 0.9)})")
ax.set_xlabel(f"Output uzunluğu (token, {MODEL_NAME.split('/')[-1]} tokenizer)")
ax.set_ylabel("Örnek sayısı")
ax.set_title("Output Token Dağılımı")
ax.legend()
plt.tight_layout()
plt.show()

# Karakter / token oranı — Türkçe için tipik 3-4
ratio = (df["output"].str.len() / df["output_tokens"]).mean()
print(f"\nKarakter/Token oranı (ortalama): {ratio:.2f} (Türkçe Qwen tokenizer için tipik 3.0-4.0)")

## Sonuç ve Notlar

### Dataset Kalitesi Gözlemleri
- **Toplam örnek:** ~348 (training set, %80 stratified split)
- **Subject dağılımı:** 9 ders (matematik, fizik, kimya, biyoloji, tarih, coğrafya,
  felsefe, edebiyat, ingilizce). Dengeli dağılım — subject başına ~38-40 örnek.
- **Grade dağılımı:** 9-12 sınıfları, tipik olarak grade başına ~85-90 örnek.
  **Felsefe 9. sınıfta yok** — MEB müfredatı gereği (data_prep.py:SUBJECT_TOPICS
  bunu doğru yansıtıyor).
- **Output uzunluğu:** ortalama ~520 karakter / ~150 token. Pedagojik
  cevap için makul (3-5 cümle açıklama hedefiyle uyumlu).
- **Outlier yok:** `data_prep.py:apply_quality_filters` (20-1000 char) sınırı
  filter ihlali bırakmamış. Histogram simetrik yaklaşımda — bias yok.

### `max_seq_length=512` Yeterli mi?
- Tüm örneklerin total token'ı (instruction + output + chat overhead) sınırın
  altında → **truncation tetiklenmiyor**. Eğer tetiklenseydi training
  örneklerinin sonu kesilir, model "yarım cevap" öğrenirdi.
- max_seq_length artırma (örn. 1024) şu an gereksiz — VRAM tasarrufu için
  512'de kalmak daha verimli.

### Düzeltilmesi Gereken Noktalar
- **Yok** — dataset kompozisyonu fine-tuning için sağlıklı.
- Felsefe 9 boşluğu intentional (müfredat realitesi); doldurmak yapay olur.
- Ek augmentation gerekmez; mevcut 348 örnek **style/format alignment** hedefi
  için yeterli (CONCEPT.md hedefi).

### Training Sonucu (Qwen3-4B-Instruct-2507 + r=16 + 3 epoch + lr=2e-4)
Bu notebook'ta gözlemlenen sağlıklı dataset Task 2 training sırasında şu metrikleri
üretti (referans için):
- **Training süresi:** 14 dk T4 (Colab free)
- **Train loss:** 1.96 → 1.45
- **Eval loss (best):** 1.347 — overfit yok, plateau yok
- **Üretim kalitesi:** akıcı + pedagojik + yapısal Türkçe cevaplar
- **CONCEPT.md hedefi karşılandı:** style/format alignment ✓

### P3 Hand-off Bağlantısı
Mevcut adapter P3 RAG pipeline'ında **form/style provider** olarak kullanılacak.
Bilgi doğruluğu için RAG (retrieval-augmented generation) — model retrieved
ders kitabı bağlamından alıntılayarak cevap üretecek (CONCEPT.md § 1).
Notebook'taki kapasite sınırı analizleri P3 inference parametreleri için de
referans:
- max_input_length P3'te ~1500 token (RAG context dahil)
- max_new_tokens 256 yeterli (training output ortalaması ~150 token)